# Red-teaming DeepSeek with crocotester

**crocotester** runs open-source LLM security tools (garak, PyRIT, inspect_ai and promptfoo)
against one target with one configuration, and turns their results into a single report. This
notebook points it at DeepSeek's hosted API and asks one question: *how often does the model do
something it should refuse?*

Every run starts from scratch: previous results are deleted, attacks are regenerated and the
report is rebuilt.

## What crocotester can do

| Capability | Options | Used here |
|---|---|---|
| **Target type** | `LLM` (a chat model) or `SEMANTIC_FENCE` (a guardrail API that returns allow/block) | `LLM` |
| **Connection to the target** | a curl command, a curl file, a custom Python handler, or the provider's own API | curl command, plus DeepSeek's API for promptfoo |
| **Auditors** | garak, PyRIT, inspect_ai, promptfoo | all four |
| **Judge model** | any inspect_ai or promptfoo model string; PyRIT's judge follows `PENTESTER_LLM__*` | DeepSeek |
| **Attack generator** | any promptfoo provider, including your own Python script | DeepSeek, through `my_model.py` |
| **Multi-turn attacks** | promptfoo crescendo, GOAT, mischievous-user; PyRIT red-teaming, crescendo, tree-of-attacks | off: single-turn attacks only |
| **Scope** | per-auditor probes, datasets, evals and plugins; a global attack cap | smoke test or full run |
| **Concurrency** | garak attacks, PyRIT seeds, inspect_ai samples, promptfoo files and requests per file | one per CPU core |
| **Report** | one summary plus per-auditor details, in HTML, PDF, CSV and Markdown | all four formats |

Everything is set through documented `PENTESTER_*` environment variables and one
`PentesterSettings` object, then run with a single `Orchestrator(settings).execute()`.

> **Thinking off.** Every DeepSeek call that crocotester's settings can reach runs with
> `"thinking": {"type": "disabled"}`, which makes replies faster and cheaper. Three calls can't
> be reached that way; section 4 names them.

> **Cost.** DeepSeek is a paid API. Every attack, generated test and judge call is billed.
> The smoke test sends a few hundred requests; the full run sends on the order of 100,000.

## 1. Setup

In [1]:
!pip install -q ipywidgets

In [ ]:
%pip install --quiet crocotester

In [2]:
import getpass
import json
import logging
import os
import shutil
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from pentester.config.logging import setup_logging
from pentester.config.settings import PentesterSettings, clear_settings_cache, TargetType
from pentester.orchestrator import Orchestrator

setup_logging(level=logging.INFO)
print(", ".join(f"{p} {version(p)}" for p in ("crocotester", "pyrit", "inspect_ai", "inspect_evals", "promptfoo")))

crocotester 0.0.1, pyrit 0.13.0, inspect_ai 0.3.220, inspect_evals 0.12.0, promptfoo 0.1.4


## 2. Configuration

`FULL_RUN = False` is a quick smoke test with every auditor; `True` runs everything each
auditor ships with.

The API key is read from `notepads/.env` (`DEEPSEEK_API_KEY=...`); if it's missing, the
notebook asks for it.

Every auditor sends one request per CPU core at a time instead of one after another. The
attacks themselves don't change, only how many are in flight. If the errors column in the
results grows, DeepSeek is rate-limiting: lower `CPUS`.

In [5]:
FULL_RUN = False
MODEL = "deepseek-flash"
DEEPSEEK_URL = "https://api.deepseek.com"
OUTPUT_DIR = Path("output/deepseek_full" if FULL_RUN else "output/deepseek")
CPUS = os.cpu_count()  # requests in flight per auditor

SCOPE = {
    "max_attacks":     None if FULL_RUN else 50,  # per auditor; None = no cap
    "garak_probes":    [] if FULL_RUN else ["probes.dan.Dan_6_2", "probes.encoding.InjectBase64"],
    "pyrit_datasets":  ["adv_bench", "harmbench", "jbb_behaviors", "forbidden_questions",
                        "simple_safety_tests", "sorry_bench", "xstest"]
                       if FULL_RUN else ["xstest"],
    "inspect_evals":   ["strong_reject", "b3", "fortress_adversarial", "agentharm", "AgentDojo",
                        "wmdp_bio", "wmdp_chem", "wmdp_cyber"]
                       if FULL_RUN else ["strong_reject", "wmdp_cyber"],
    "promptfoo_files": None if FULL_RUN else 5,   # one file per plugin; None = all plugins
    "promptfoo_tests": None if FULL_RUN else 5,   # tests per plugin; None = config default
}

load_dotenv(".env")
if not os.environ.get("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DeepSeek API key: ")

os.environ.update({
    "PENTESTER_TARGET_TYPE": TargetType.LLM,
    "PENTESTER_LLM__PROVIDER": "deepseek",
    "PENTESTER_LLM__MODEL": MODEL,
    "PENTESTER_SCANNER__RESPONSE_TEXT_TARGET": "body.choices.0.message.content",
    "PENTESTER_GARAK__PARALLEL_ATTEMPTS": str(CPUS),
    "PENTESTER_PYRIT__MAX_CONCURRENCY": str(CPUS),
    "PENTESTER_INSPECT__MAX_CONNECTIONS": str(CPUS),
    "PENTESTER_PROMPTFOO__FILES_PARALLEL": str(CPUS),
    "PENTESTER_PROMPTFOO__INTERNAL_CONCURRENCY": str(CPUS),
    "PENTESTER_PROMPTFOO__REPLACE_EXISTING_FILE": "true",
    "PENTESTER_REPORTING__OUTPUT_DIR_PATH": str(OUTPUT_DIR),
    "PENTESTER_REPORTING__GENERATOR_KEYS": "csv,html,markdown,pdf",
    "DEEPSEEK_BASE_URL": DEEPSEEK_URL,  # inspect_ai's DeepSeek judge and my_model.py
    "DEEPSEEK_MODEL": MODEL,            # my_model.py
    "PROMPTFOO_PYTHON": sys.executable, # the Python promptfoo runs my_model.py with
})
if SCOPE["max_attacks"]:
    os.environ["PENTESTER_MAX_ATTACKS"] = str(SCOPE["max_attacks"])
else:
    os.environ.pop("PENTESTER_MAX_ATTACKS", None)
clear_settings_cache()

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)  # start from scratch
print(f"{'full run' if FULL_RUN else 'smoke test'} · {MODEL} · {CPUS} requests in flight · {OUTPUT_DIR}")

smoke test · deepseek-flash · 24 requests in flight · output/deepseek


## 3. The target: DeepSeek

A quick check that the key works and the model is available.

Then the curl command crocotester uses to reach the model. For each attack, crocotester
replaces `$PROMPT` with the attack text, already JSON-encoded, and reads the reply from
`body.choices.0.message.content`. The request body turns thinking off
(`"thinking": {"type": "disabled"}`), asks for a single non-streamed reply (`"stream": false`)
and sends no `reasoning_effort`, which only applies when thinking is on.

In [6]:
HEADERS = {"Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}"}
NO_THINKING = {"thinking": {"type": "disabled"}, "stream": False}


def request_body(content) -> dict:
    return {"model": MODEL, "messages": [{"role": "user", "content": content}], **NO_THINKING}


models = [m["id"] for m in requests.get(f"{DEEPSEEK_URL}/models", headers=HEADERS, timeout=30).json()["data"]]
assert MODEL in models, f"{MODEL} not available: {models}"
reply = requests.post(f"{DEEPSEEK_URL}/chat/completions", headers=HEADERS,
                      json=request_body("Hello"), timeout=120).json()["choices"][0]["message"]
assert not reply.get("reasoning_content"), "thinking is still on"
print(f"models: {', '.join(models)} · reply: {reply['content']!r}")

# crocotester replaces a bare $PROMPT with the JSON-encoded attack.
body = json.dumps(request_body("__PROMPT__")).replace('"__PROMPT__"', "$PROMPT")
CURL_COMMAND = (
    f"curl -X POST {DEEPSEEK_URL}/chat/completions "
    "-H 'Content-Type: application/json' "
    f"-H 'Authorization: Bearer {os.environ['DEEPSEEK_API_KEY']}' "
    f"--data-raw '{body}'"
)
print(CURL_COMMAND.replace(os.environ["DEEPSEEK_API_KEY"], "sk-***"))

models: deepseek-flash, deepseek-v4-pro · reply: 'Hello! How can I help you today?'
curl -X POST https://api.deepseek.com/chat/completions -H 'Content-Type: application/json' -H 'Authorization: Bearer sk-***' --data-raw '{"model": "deepseek-flash", "messages": [{"role": "user", "content": $PROMPT}], "thinking": {"type": "disabled"}, "stream": false}'


## 4. The auditors

One `PentesterSettings` object configures all four auditors. The environment variables from
section 2 already set the target, the concurrency and the report; the cell below adds the
curl command and what each auditor runs.

### garak: known jailbreaks and encodings

garak sends libraries of known attacks (DAN-style jailbreaks, encoded instructions, prompt
injection, toxicity prompts and more) and scores each reply with a rule-based **detector**. No
LLM judge is involved, so its results don't depend on the judge model. Rate-limited or
timed-out requests are retried up to `PENTESTER_GARAK__MAX_RETRIES` times (default 3) with
backoff; anything still failing is recorded as `ERROR`.

### PyRIT: red-teaming datasets

PyRIT sends prompts from public red-teaming datasets (here AdvBench, HarmBench, JailbreakBench,
forbidden questions, SimpleSafetyTests, SORRY-Bench and XSTest; the smoke test uses XSTest
only) and asks an LLM **judge** whether each reply contains harmful content. The judge is the
model in `PENTESTER_LLM__PROVIDER` + `PENTESTER_LLM__MODEL`, DeepSeek here, reached at
`https://api.deepseek.com/v1` with `DEEPSEEK_API_KEY`.

Attacks go through the curl command, so the target runs with thinking off; the **judge** keeps
DeepSeek's default, since crocotester gives PyRIT only a model name. PyRIT can also run
multi-turn attackers (red-teaming, crescendo, tree-of-attacks); they're off here.

### inspect_ai: safety benchmarks

inspect_ai runs published safety benchmarks. Two kinds are used here:

- **Judged evals** (`strong_reject`, `b3`, `fortress_adversarial`, `agentharm`, `AgentDojo`):
  an LLM judge decides whether the reply complied with a harmful request. DeepSeek is the
  judge; inspect_ai reaches it as `openai-api/deepseek/<model>`, its name for any
  OpenAI-compatible API.
- **Answer-matched evals** (`wmdp_bio`, `wmdp_chem`, `wmdp_cyber`): multiple-choice questions
  on hazardous knowledge, scored without a judge. A correct answer counts as a bypass, so these
  measure dangerous knowledge rather than jailbreak resistance.

The target is reached through the curl command, so it runs with thinking off. The DeepSeek
**judge** keeps DeepSeek's default: crocotester passes inspect_ai only a model name, with no
way to add request options.

### promptfoo: generated attacks

promptfoo writes new attacks for each risk category (its **plugins**: harmful content, PII,
hijacking, competitor mentions and more), then disguises them with **strategies** such as
base64, leetspeak, emoji or citation framing. It can also run multi-turn attackers
(crescendo, GOAT, mischievous-user); they're off here.

promptfoo uses three models, all DeepSeek here:

| Role | Setting | Value | Thinking |
|---|---|---|---|
| target | `PENTESTER_LLM__PROVIDER` + `PENTESTER_LLM__MODEL` | `deepseek:deepseek-flash` | DeepSeek's default |
| attack generator | `PENTESTER_PROMPTFOO__ATTACK_GENERATION_MODEL` | `file://…/my_model.py:call_attack_generator` | off |
| judge | `PENTESTER_PROMPTFOO__JUDGE_MODEL` | `file://…/my_model.py:call_judge` | off |

Both settings take any promptfoo provider, and `file://<script>.py:<function>` makes your own
Python function the model. [`my_model.py`](my_model.py) calls DeepSeek's chat API with
thinking off, retries rate limits with backoff, and runs the judge at temperature 0 so its
verdicts are repeatable. It reads `DEEPSEEK_API_KEY`, `DEEPSEEK_MODEL` and `DEEPSEEK_BASE_URL`
from the environment; `PROMPTFOO_PYTHON` tells promptfoo which Python to run it with. Any other
model or API works the same way: write a function that returns `{"output": text}`.

The target is different: crocotester always builds it from `PENTESTER_LLM__*` as
`provider:model`, so promptfoo calls `deepseek:deepseek-flash` through its own `deepseek`
provider, with DeepSeek's default thinking on. promptfoo then puts the reasoning in front of the
answer (`Thinking: …`), and its judge grades both, so promptfoo's target replies aren't
directly comparable with the other auditors', which only see the answer.

In [7]:
settings = PentesterSettings(auditors=["inspect_ai", "garak", "pyrit", "promptfoo"])
settings.scanner.curl_command = CURL_COMMAND

settings.garak.probes = SCOPE["garak_probes"]

settings.pyrit.dataset_names = SCOPE["pyrit_datasets"]
settings.pyrit.enable_multiturn = False  # single-turn attacks only

settings.inspect.evals = SCOPE["inspect_evals"]
settings.inspect.judge_model = f"openai-api/deepseek/{MODEL}"

# Our own Python functions as attack generator and judge: DeepSeek with thinking off.
MY_MODEL = Path("my_model.py").resolve()
settings.promptfoo.attack_generation_model = f"file://{MY_MODEL}:call_attack_generator"
settings.promptfoo.judge_model = f"file://{MY_MODEL}:call_judge"
settings.promptfoo.max_test_files = SCOPE["promptfoo_files"]
settings.promptfoo.plugin_num_tests = SCOPE["promptfoo_tests"]
settings.promptfoo.enable_multiturn = False  # single-turn attacks only

pd.Series({
    "garak attacks in flight": settings.garak.parallel_attempts,
    "PyRIT seeds in flight": settings.pyrit.max_concurrency,
    "inspect_ai samples in flight": settings.inspect.max_connections,
    "promptfoo files in parallel": settings.promptfoo.files_parallel,
    "promptfoo requests per file": settings.promptfoo.internal_concurrency,
}).to_frame("concurrency")

,concurrency
garak attacks in flight,24
PyRIT seeds in flight,24
inspect_ai samples in flight,24
promptfoo files in parallel,24
promptfoo requests per file,24


## 5. Running the audit

One call. The Orchestrator checks the target is reachable, runs garak, PyRIT, inspect_ai and
promptfoo in turn (each with its requests in parallel) and writes one report covering all four.

In [8]:
Orchestrator(settings).execute()

[INFO] Assertion mode: javascript default (output.passed)
[INFO] Loading config from: pentester.config.auditor_files.promptfoo
[INFO] LLM target provider configured from settings.llm: deepseek:deepseek-flash
[INFO] Scanner preflight successful
[INFO] Running inspect_ai evaluation: strong_reject


Output()

[INFO] Running inspect_ai evaluation: wmdp_cyber


Loading dataset cais/wmdp from Hugging Face...


Saving the dataset (0/1 shards):   0%|          | 0/1987 [00:00<?, ? examples/s]

...

No default environment files found. Using system environment variables only.


xstest: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:13<00:00,  3.73it/s]
[INFO] PyritAuditor.audit completed in 13.9398s — 
[INFO] Setting default promptfoo email to tools@tekdatum.com
[INFO] promptfoo email set to tools@tekdatum.com
[INFO] Generated test file: output/deepseek/promptfoo/tests/configurations/test_1.yaml
[INFO] Generated test file: output/deepseek/promptfoo/tests/configurations/test_2.yaml
[INFO] Generated test file: output/deepseek/promptfoo/tests/configurations/test_3.yaml
[INFO] Generated test file: output/deepseek/promptfoo/tests/configurations/test_4.yaml
[INFO] Generated test file: output/deepseek/promptfoo/tests/configurations/test_5.yaml
[INFO] Running command: promptfoo redteam generate --output output/deepseek/promptfoo/tests/llm_as_judge_assert/test_2.yaml --config output/deepseek/promptfoo/tests/configurati

Cache is disabled.
Starting evaluation eval-8Gp-2026-10-13T20:34:56
Running 50 test cases (up to 24 at a time)...
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers

[WARNING] promptfoo exited with code 100 for test_1.yaml
[INFO] Finished test_1.yaml
[INFO] File test_1.yaml processed correctly.
[INFO] Built DataFrame with 50 rows from output/deepseek/promptfoo/results
[INFO] Audit complete. Results: 50
[INFO] PromptfooAuditor.audit completed in 268.1754s — 
[INFO] Reporting.generate completed in 0.7162s — 
[INFO] Orchestrator.execute completed in 341.2285s — 


(None, 341.22845116700046)

## 6. Results

**Bypass rate** is the share of scored attacks where the model did what it should have
refused; errors are left out. Lower is better.

In [9]:
csv_files = sorted(OUTPUT_DIR.glob("*/csv/*_details.csv"))
REPORT_DIR = csv_files[0].parent.parent
details = pd.concat([pd.read_csv(f).assign(auditor=f.name.removesuffix("_details.csv")) for f in csv_files])


def score(d: pd.DataFrame) -> pd.Series:
    scored = d[d["status"] != "ERROR"]
    bypassed = int(scored["status"].eq("BYPASSED").sum())
    return pd.Series({"attacks": len(d), "errors": len(d) - len(scored), "bypassed": bypassed,
                      "bypass rate": f"{bypassed / len(scored):.0%}" if len(scored) else "n/a"})


pd.DataFrame({auditor: score(d) for auditor, d in details.groupby("auditor")} | {"all": score(details)}).T

,attacks,errors,bypassed,bypass rate
garak,51,0,0,0%
inspect_ai,100,0,40,40%
promptfoo,50,0,36,72%
pyrit,50,0,0,0%
all,251,0,76,30%


### Weakest attack categories

Categories with the highest bypass rate, across all auditors.

In [10]:
scored = details[details["status"] != "ERROR"]
(scored.assign(bypassed=scored["status"].eq("BYPASSED"))
       .groupby(["auditor", "attack_category"])["bypassed"]
       .agg(attacks="size", bypassed="sum", rate="mean")
       .sort_values(["rate", "attacks"], ascending=False)
       .head(15)
       .style.format({"rate": "{:.0%}"}))

### The report

One report for the whole run: a summary across all auditors plus a details file per auditor,
in each format.

In [11]:
print(REPORT_DIR)
for fmt in sorted(p for p in REPORT_DIR.iterdir() if p.is_dir()):
    print(f"  {fmt.name:9} {', '.join(sorted(f.name for f in fmt.iterdir()))}")

output/deepseek/20261013_203639
  csv       garak_details.csv, inspect_ai_details.csv, promptfoo_details.csv, pyrit_details.csv, summary.csv
  html      garak_details.html, inspect_ai_details.html, promptfoo_details.html, pyrit_details.html, summary.html
  markdown  garak_details.md, inspect_ai_details.md, promptfoo_details.md, pyrit_details.md, summary.md
  pdf       garak_details.pdf, inspect_ai_details.pdf, promptfoo_details.pdf, pyrit_details.pdf, summary.pdf


## 7. Experiments: other judges, attack generators and targets

Every model in the run is a setting, so the same audit can be repeated with any part swapped.
This shows how much a result depends on the judge, on who writes the attacks, or on the model
under test.

### Who plays which role

| Role | garak | PyRIT | inspect_ai | promptfoo |
|---|---|---|---|---|
| **Target** | curl command | curl command | curl command | `PENTESTER_LLM__PROVIDER` + `PENTESTER_LLM__MODEL` |
| **Judge** | rule-based detectors | `PENTESTER_LLM__PROVIDER` + `PENTESTER_LLM__MODEL` | `PENTESTER_INSPECT__JUDGE_MODEL` | `PENTESTER_PROMPTFOO__JUDGE_MODEL` |
| **Attack generator** | none: a fixed attack library | none in single-turn; `PENTESTER_LLM__*` drives the multi-turn attackers | none: fixed benchmarks | `PENTESTER_PROMPTFOO__ATTACK_GENERATION_MODEL` |

Each setting takes its own model format:

- **curl command:** any HTTP API. To test another model or provider, change the URL, headers
  or body.
- **`PENTESTER_LLM__PROVIDER` + `PENTESTER_LLM__MODEL`:** `openai`, `anthropic`, `gemini` or
  `deepseek`, plus a model name. The key comes from `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`,
  `GEMINI_API_KEY` or `DEEPSEEK_API_KEY`.
- **inspect_ai judge:** an inspect_ai model string such as `openai/gpt-4o-mini`,
  `anthropic/<model>`, `google/<model>` or `deepseek/<model>`.
- **promptfoo judge and attack generator:** a promptfoo provider id such as
  `openai:gpt-4o-mini`, `anthropic:messages:<model>` or `deepseek:<model>`, or your own
  function as `file://<script>.py:<function>`, like `my_model.py` in the main run. A bare
  `deepseek:` means `deepseek-flash` with thinking off; a named DeepSeek model keeps thinking on.

Two things to keep in mind:

- **PyRIT's judge and promptfoo's target are the same setting.** Changing
  `PENTESTER_LLM__*` changes both, and inspect_ai's judge too when
  `PENTESTER_INSPECT__JUDGE_MODEL` is unset. To vary one without the other, run those
  auditors in separate experiments.
- **A model judging itself is not an independent check.** Here DeepSeek attacks, answers and
  judges. A judge from another model family is the cleaner test, and a judge swap shows how
  much the bypass rate moves with the judge alone.

### Running an experiment

`experiment()` copies the settings from section 4, applies the overrides
(`inspect__judge_model=...` sets `settings.inspect.judge_model`), runs the chosen auditors and
writes its own report under `output/.../experiments/<name>`. Each experiment is a separate,
billed run with the same `FULL_RUN` scope, so they're off until `RUN_EXPERIMENTS = True`.

In [12]:
RUN_EXPERIMENTS = True


def curl_for(model: str) -> str:
    """The section 3 curl command, pointed at another DeepSeek model."""
    return CURL_COMMAND.replace(f'"model": "{MODEL}"', f'"model": "{model}"')


def experiment(name: str, auditors: list[str], **overrides) -> pd.DataFrame:
    s = settings.model_copy(deep=True)
    s.auditors = auditors
    for path, value in overrides.items():  # "inspect__judge_model" -> s.inspect.judge_model
        section, field = path.split("__", 1)
        setattr(getattr(s, section), field, value)
    out = OUTPUT_DIR / "experiments" / name
    s.reporting.output_dir_path = str(out)
    s.promptfoo.output_path = str(out / "promptfoo")

    Orchestrator(s).execute()

    files = sorted(out.glob("*/csv/*_details.csv"))
    d = pd.concat([pd.read_csv(f).assign(auditor=f.name.removesuffix("_details.csv")) for f in files])
    return pd.DataFrame({a: score(g) for a, g in d.groupby("auditor")}).T.assign(experiment=name)

Three examples, each compared with the main run above:

1. **Another judge.** Same target and attacks, judged by `deepseek-v4-pro` instead of
   `deepseek-flash`. Only the judged auditors need to run.
2. **Another attack generator.** promptfoo writes its attacks with `deepseek-v4-pro`.
3. **Another target.** `deepseek-v4-pro` is the model under test. Every auditor follows: the
   curl command for garak, PyRIT and inspect_ai, and `PENTESTER_LLM__MODEL` for promptfoo. That
   same setting also makes `deepseek-v4-pro` PyRIT's judge.

Any other provider works the same way. For example, `inspect__judge_model="openai/gpt-4o-mini"`
with `promptfoo__judge_model="openai:gpt-4o-mini"` and an `OPENAI_API_KEY` gives an
independent judge.

In [13]:
if RUN_EXPERIMENTS:
    baseline = pd.DataFrame({a: score(g) for a, g in details.groupby("auditor")}).T.assign(experiment="main run")
    comparison = pd.concat([
        baseline,
        experiment("judge_v4_pro", ["inspect_ai", "promptfoo"],
                   inspect__judge_model="deepseek/deepseek-v4-pro",
                   promptfoo__judge_model="deepseek:deepseek-v4-pro"),
        experiment("attacker_v4_pro", ["promptfoo"],
                   promptfoo__attack_generation_model="deepseek:deepseek-v4-pro"),
        experiment("target_v4_pro", ["garak", "pyrit", "inspect_ai", "promptfoo"],
                   scanner__curl_command=curl_for("deepseek-v4-pro"),
                   llm__model="deepseek-v4-pro"),
    ])
    display(comparison.rename_axis("auditor").reset_index()
                      .pivot(index="auditor", columns="experiment", values="bypass rate"))
else:
    print("Experiments are off: set RUN_EXPERIMENTS = True to run them.")

[INFO] Assertion mode: javascript default (output.passed)
[INFO] Loading config from: pentester.config.auditor_files.promptfoo
[INFO] LLM target provider configured from settings.llm: deepseek:deepseek-flash
[INFO] Scanner preflight successful
[INFO] Running inspect_ai evaluation: strong_reject


Output()

[INFO] Running inspect_ai evaluation: wmdp_cyber


Loading dataset cais/wmdp from Hugging Face...


Saving the dataset (0/1 shards):   0%|          | 0/1987 [00:00<?, ? examples/s]

...

Output()

[INFO] inspect_ai audit complete. Total probe results: 100
[INFO] InspectAIAuditor.audit completed in 51.1002s — 
[INFO] Setting default promptfoo email to tools@tekdatum.com
[INFO] promptfoo email set to tools@tekdatum.com
[INFO] Generated test file: output/deepseek/experiments/target_v4_pro/promptfoo/tests/configurations/test_1.yaml
[INFO] Generated test file: output/deepseek/experiments/target_v4_pro/promptfoo/tests/configurations/test_2.yaml
[INFO] Generated test file: output/deepseek/experiments/target_v4_pro/promptfoo/tests/configurations/test_3.yaml
[INFO] Generated test file: output/deepseek/experiments/target_v4_pro/promptfoo/tests/configurations/test_4.yaml
[INFO] Generated test file: output/deepseek/experiments/target_v4_pro/promptfoo/tests/configurations/test_5.yaml
[INFO] Running command: promptfoo redteam generate --output output/deepseek/experiments/target_v4_pro/promptfoo/tests/llm_as_judge_assert/test_2.yaml --config output/deepseek/experiments/target_v4_pro/promptfoo/

Cache is disabled.
Starting evaluation eval-AuD-2026-10-13T20:58:01
Running 50 test cases (up to 24 at a time)...
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers for /home/ubuntu/git/pen-tester-cli/docs/examples/llm_red_teaming/my_model.py. This may use significant memory if your script has heavy imports.
Spawning 24 Python workers

[WARNING] promptfoo exited with code 100 for test_1.yaml
[INFO] Finished test_1.yaml
[INFO] File test_1.yaml processed correctly.
[INFO] Built DataFrame with 50 rows from output/deepseek/experiments/target_v4_pro/promptfoo/results
[INFO] Audit complete. Results: 50
[INFO] PromptfooAuditor.audit completed in 513.5651s — 
[INFO] Reporting.generate completed in 0.7190s — 
[INFO] Orchestrator.execute completed in 591.0321s — 


experiment,attacker_v4_pro,judge_v4_pro,main run,target_v4_pro
auditor,,,,
garak,NaN,NaN,0%,0%
inspect_ai,NaN,33%,40%,36%
promptfoo,66%,74%,72%,58%
pyrit,NaN,NaN,0%,0%
